**Install Required Packages**

In [ ]:
!pip install google-cloud-bigquery google-cloud-storage google-auth


**Import libraries and initialize client instances**


In [16]:
import os
import asyncio
import google.auth

os.environ['GOOGLE_CLOUD_PROJECT']

GOOGLE_CLOUD_PROJECT = "us-gcp-ame-con-e5556-npd-1"
GOOGLE_CLOUD_LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

os.environ['GOOGLE_CLOUD_PROJECT'] = GOOGLE_CLOUD_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = GOOGLE_CLOUD_LOCATION
os.environ['MODEL_NAME'] = MODEL_NAME

google.auth.default()

(<google.auth.compute_engine.credentials.Credentials at 0x7e34b78277d0>,
 'us-gcp-ame-con-e5556-npd-1')

In [17]:
os.getenv("GOOGLE_CLOUD_PROJECT")
os.getenv("GOOGLE_CLOUD_LOCATION")

'us-central1'

In [18]:
from google.cloud import bigquery
from google.cloud import storage

# Initialize BigQuery and Storage clients
bigquery_client = bigquery.Client()
storage_client = storage.Client()

# --- Configuration for your data load ---
# Replace with your project ID, dataset ID, table ID, and GCS bucket details
project_id = "us-gcp-ame-con-e5556-npd-1"
location = "us-central1"
dataset_id = "emergency_response"
raw_table_id = "airports_data_raw"
gold_table_id = "airports_alerts_data"
model_id = "public_communication_model"
source_bucket_name = "gs://labs.roitraining.com/data-to-ai-workshop"
source_file_name = "gs://labs.roitraining.com/data-to-ai-workshop/airports.csv"

**Create Dataset**

In [4]:
sql_ddl_dataset_statement = f"""
CREATE SCHEMA IF NOT EXISTS `{project_id}.{dataset_id}`
OPTIONS(
    location = '{location}',
    default_table_expiration_days=14
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_dataset_statement)

# Wait for the job to complete
query_job.result()

print(f"Dataset '{dataset_id}' created or already exists in project '{project_id}' at location '{location}'.")


Dataset 'emergency_response' created or already exists in project 'us-gcp-ame-con-e5556-npd-1' at location 'us-central1'.


**Load data from Storage bucket into Big Query Table**

In [5]:
raw_table_ref = f"{project_id}.{dataset_id}.{raw_table_id}"

# Configure the load job
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Assuming the CSV has a header row
    autodetect=True,      # Let BigQuery autodetect schema
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Overwrite if table exists
)

# Start the load job
load_job = bigquery_client.load_table_from_uri(
    source_file_name, raw_table_ref, job_config=job_config
)

# Wait for the job to complete
load_job.result()

print(f"Data loaded from '{source_file_name}' into BigQuery table '{raw_table_ref}'.")

Data loaded from 'gs://labs.roitraining.com/data-to-ai-workshop/airports.csv' into BigQuery table 'us-gcp-ame-con-e5556-npd-1.emergency_response.airports_data_raw'.


In [6]:
df = bigquery_client.list_rows(f"{raw_table_ref}").to_dataframe()
df


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11,NA,US,US-PA,Bensalem,False,None,None,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,None,None
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435,NA,US,US-KS,Leoti,False,None,None,00AA,00AA,None,None,None
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450,NA,US,US-AK,Anchor Point,False,None,None,00AK,00AK,None,None,None
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820,NA,US,US-AL,Harvest,False,None,None,00AL,00AL,None,None,None
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80,NA,US,US-AK,King Salmon,False,None,None,00AN,00AN,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82888,32753,ZYYY,medium_airport,Shenyang Dongta Airport,41.784401,123.496002,<NA>,AS,CN,CN-21,"Dadong, Shenyang",False,None,None,ZYYY,None,None,None,None
82889,46378,ZZ-0001,heliport,Sealand Helipad,51.894444,1.482500,40,EU,GB,GB-ENG,Sealand,False,None,None,None,None,http://www.sealandgov.org/,https://en.wikipedia.org/wiki/Principality_of_...,Roughs Tower Helipad
82890,307326,ZZ-0002,small_airport,Glorioso Islands Airstrip,-11.584278,47.296389,11,AF,TF,TF-U-A,Grande Glorieuse,False,None,None,None,None,None,None,None
82891,346788,ZZ-0003,small_airport,Fainting Goat Airport,32.110587,-97.356312,690,NA,US,US-TX,Blum,False,None,None,87TX,87TX,None,None,None


In [10]:
df.groupby('type')['name'].unique()


,name
type,
balloonport,"[Clinton Elks Lodge Balloonport, Aeronut Park ..."
closed,"[Ring Hill Airport, Land's Field, Flying S Ran..."
heliport,"[Total RF Heliport, Kitchen Creek Helibase Hel..."
large_airport,"[Honiara International Airport, Port Moresby J..."
medium_airport,"[Aleknagik / New Airport, Sas Al Nakheel Air B..."
seaplane_base,"[Cliche Cove Seaplane Base, Annapolis Seaplane..."
small_airport,"[Aero B Ranch Airport, Lowell Field, Epps Airp..."


**Filter Large Airports in US**

In [12]:
large_airports_us = df[(df['type'] == 'large_airport') & (df['iso_country'] == 'US')]
large_airports_us


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
37412,16091,KABQ,large_airport,Albuquerque International Sunport,35.039976,-106.608925,5355,NA,US,US-NM,Albuquerque,True,KABQ,ABQ,KABQ,ABQ,http://www.abqsunport.com/,https://en.wikipedia.org/wiki/Albuquerque_Inte...,None
37431,3364,KADW,large_airport,Joint Base Andrews,38.810799,-76.866997,280,NA,US,US-MD,Camp Springs,False,KADW,ADW,KADW,ADW,http://www.jba.af.mil/,https://en.wikipedia.org/wiki/Joint_Base_Andrews,Andrews Air Force Base
37532,3384,KATL,large_airport,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101,1026,NA,US,US-GA,Atlanta,True,KATL,ATL,KATL,ATL,http://www.atlanta-airport.com/,https://en.wikipedia.org/wiki/Hartsfield–Jacks...,None
37541,3386,KAUS,large_airport,Austin Bergstrom International Airport,30.197535,-97.662015,542,NA,US,US-TX,Austin,True,KAUS,AUS,KAUS,AUS,http://www.ci.austin.tx.us/austinairport/,https://en.wikipedia.org/wiki/Austin-Bergstrom...,"BSM, KBSM, Bergstrom AFB"
37588,3396,KBDL,large_airport,Bradley International Airport,41.938510,-72.688066,173,NA,US,US-CT,Hartford,True,KBDL,BDL,KBDL,BDL,http://www.bradleyairport.com/,https://en.wikipedia.org/wiki/Bradley_Internat...,"HFD, Hartford"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41558,3926,KTPA,large_airport,Tampa International Airport,27.975500,-82.533203,26,NA,US,US-FL,Tampa,True,KTPA,TPA,KTPA,TPA,https://www.tampaairport.com/,https://en.wikipedia.org/wiki/Tampa_Internatio...,None
41577,3930,KTUL,large_airport,Tulsa International Airport,36.198399,-95.888100,677,NA,US,US-OK,Tulsa,True,KTUL,TUL,KTUL,TUL,None,https://en.wikipedia.org/wiki/Tulsa_Internatio...,None
51309,5388,PANC,large_airport,Ted Stevens Anchorage International Airport,61.179004,-149.992561,152,NA,US,US-AK,Anchorage,True,PANC,ANC,PANC,ANC,https://dot.alaska.gov/anc/,https://en.wikipedia.org/wiki/Ted_Stevens_Anch...,None
52380,5453,PHNL,large_airport,Daniel K Inouye International Airport,21.320620,-157.924228,13,NA,US,US-HI,"Honolulu, Oahu",True,PHNL,HNL,PHNL,HNL,http://airports.hawaii.gov/hnl/,https://en.wikipedia.org/wiki/Daniel_K._Inouye...,"Hickam Air Force Base, HIK, PHIK, KHNL, Honolu..."


**Retrieve Forecast URLs for each airport and append it to the dataframe**

In [31]:
import requests
headers={'userAgent':'us-airport-weather-forecast', 'accept': 'application/geo+json'}
def get_forecast_url(latitude, longitude):
    """
    Retrieves the forecast URL for a given latitude and longitude using the
    api.weather.gov points endpoint.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.

    Returns:
        str: The forecast URL, or None if an error occurs.
    """

    url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        response = requests.get(url,headers=headers)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = response.json()
        forecast_url = data.get("properties", {}).get("forecast")
        print(f"Forecast URL for {latitude},{longitude}: {forecast_url}")
        if forecast_url:
            return forecast_url
        else:
            return ""
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving forecast URL for {latitude},{longitude}: {e}")
        return None

# Apply the function to the large_airports_us dataframe
# large_airports_us['forecast_url'] = large_airports_us.apply(
#     lambda row: get_forecast_url(row['latitude_deg'], row['longitude_deg']),
#     axis=1
# )
forecast_url = get_forecast_url(40.070985, -74.933689)
print(forecast_url)
print("Forecast URLs appended to the dataframe.")
# print(large_airports_us[['name', 'forecast_url']].head())


KeyboardInterrupt: 